<a href="https://colab.research.google.com/github/Ganeshkumarkomarneni-2005/steamvideogameanalysis/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 01: Data Understanding & Cleaning
**Project**: Steam Game Intelligence  
**Notebook**: `notebooks/01_data_cleaning.ipynb`  
**Objective**: Profile schemas, audit join quality, handle missing/invalid values, process the large reviews CSV in chunks, and export clean datasets to `data/processed/`.


In [ ]:
import os
import zipfile
import re
import pandas as pd
import numpy as np

zip_path = '../archive.zip' if os.path.exists('../archive.zip') else 'archive.zip'
z = zipfile.ZipFile(zip_path)
print(f"Archive contents: {z.namelist()}")


### 1. Games Description Dataset Profiling & Cleaning


In [ ]:
df_desc = pd.read_csv(z.open('games_description.csv'))
print(f"Raw shape: {df_desc.shape}")

def extract_num(val):
    if pd.isna(val): return 0
    nums = re.findall(r'[\d,]+', str(val))
    clean_nums = [int(n.replace(',', '')) for n in nums if n.replace(',', '').isdigit()]
    return max(clean_nums) if clean_nums else 0

def normalize_string(val):
    if pd.isna(val): return ""
    val = str(val).lower().strip()
    val = re.sub(r'[^\w\s]', '', val)
    val = re.sub(r'\s+', ' ', val)
    return val

df_desc['normalized_game_name'] = df_desc['name'].apply(normalize_string)
df_desc['release_date_clean'] = pd.to_datetime(df_desc['release_date'], errors='coerce')
df_desc['number_of_reviews_from_purchased_people_clean'] = df_desc['number_of_reviews_from_purchased_people'].apply(extract_num)
df_desc['number_of_english_reviews_clean'] = df_desc['number_of_english_reviews'].apply(extract_num)
df_desc['missing_short_desc'] = df_desc['short_description'].isna()

df_desc[['name', 'number_of_reviews_from_purchased_people_clean', 'number_of_english_reviews_clean']].head(5)
